# Q-Mini — Training-Ready Model Builder (Build now, train later)

This notebook prepares one complete **152M-class, training-ready Q-Mini package** in Google Drive.

It will:
- use the repository's from-scratch CausalConv + gated diagonal SSM + sparse Top-1 MoE model;
- keep Transformer self-attention and Q/K/V attention out of the architecture;
- fit the Hugging Face Tokenizers byte-level BPE tokenizer to the corpus you place in Drive;
- create fingerprinted train/validation token streams and the required loss masks;
- build the full random-initialized model and export Q2-v2 packed matrix weights;
- verify the model, tokenizer, data manifests and Drive copy before publishing the package.

**This notebook does not train the neural model.** It does not create an optimizer, run backward passes or update model weights. Tokenizer fitting and dataset preparation are preprocessing, not neural-model training.

The default 152M-class profile is the Colab training target. The repository's 5B profile can be initialized as a packed artifact by the separate 5B builder, but the current trainer does **not** implement FSDP/ZeRO or 5B multi-GPU training. Do not use a 5B artifact with this single-device training path.

In [ ]:
# ---------------- USER SETTINGS ----------------
MODEL_CONFIG_REL = "configs/qmini_research_152m.json"
DATA_REL = "QMini/datasets"
ART_REL = "QMini"

VOCAB_SIZE = 8192
VALIDATION_FRACTION = 0.02
NEAR_DUPLICATE_HAMMING_DISTANCE = 0
LOSS_MASK_MODE = "auto"  # auto = assistant-only for chat/Q&A, full LM for plain text

# Optional strict dataset-release gate. Supply a qmini-source-provenance-v1 JSON
# file in Google Drive before setting RELEASE_MODE=True.
SOURCE_METADATA_REL = None
RELEASE_MODE = False

# Building is the default. Change this only in a later session when you are
# intentionally ready to start neural-model training.
RUN_TRAINING_NOW = False
TRAINING_STEPS = 10000       # total optimizer-step target, not additional steps
TRAIN_BATCH_SIZE = 1
TRAIN_SEQUENCE_LENGTH = 1024
TRAIN_GRAD_ACCUM = 8
TRAIN_LEARNING_RATE = 3e-4
TRAIN_WEIGHT_DECAY = 0.1
TRAIN_EVAL_EVERY = 100
TRAIN_SAVE_EVERY = 100
TRAIN_EVAL_BATCHES = 8
TRAIN_MIN_FREE_GPU_GIB = 8.0

print("Model profile:", MODEL_CONFIG_REL)
print("Neural model training enabled:", RUN_TRAINING_NOW)
print("Architecture: causal convolution + diagonal SSM + sparse Top-1 MoE")
print("Transformer self-attention: disabled / absent by design")

In [ ]:
# ---------------- GOOGLE DRIVE + SOURCE CHECKOUT ----------------
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import shutil
import subprocess
import sys
import time

MY_DRIVE = Path("/content/drive/MyDrive")
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive is not mounted at /content/drive/MyDrive.")

DATA_DIR = MY_DRIVE / DATA_REL
ART_ROOT = MY_DRIVE / ART_REL
DATA_DIR.mkdir(parents=True, exist_ok=True)
ART_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/Q-Mini-Training-Ready")
REPOSITORY_URL = "https://github.com/lion36508/Q-Mini.git"

def run_checked(command, *, cwd=None, capture_output=False):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        capture_output=capture_output,
    )

if REPO_DIR.exists():
    run_checked(["git", "fetch", "--depth", "1", "origin", "main"], cwd=REPO_DIR)
    run_checked(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR)
else:
    run_checked(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, REPO_DIR])

SOURCE_REVISION = run_checked(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True
).stdout.strip()

MODEL_CONFIG_PATH = REPO_DIR / MODEL_CONFIG_REL
if not MODEL_CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Model config not found: {MODEL_CONFIG_PATH}")

print("Source revision:", SOURCE_REVISION)
print("Corpus folder:", DATA_DIR)
print("Drive artifact root:", ART_ROOT)

In [ ]:
# ---------------- INSTALL OPEN-SOURCE DEPENDENCIES ----------------
# Keep Colab's preinstalled CUDA-enabled PyTorch wheel. Do not replace it.
run_checked([
    sys.executable, "-m", "pip", "install",
    "numpy>=1.24", "tokenizers>=0.15", "pyarrow>=15.0", "packaging>=23",
])
run_checked([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], cwd=REPO_DIR)

import torch
from packaging.version import Version

torch_version = Version(torch.__version__.split("+", 1)[0])
if torch_version < Version("2.10.0"):
    raise RuntimeError(
        f"Q-Mini requires PyTorch >=2.10.0; Colab has {torch.__version__}. "
        "Select a compatible runtime instead of blindly replacing the CUDA wheel."
    )

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Build-only mode is CPU-compatible; later neural-model training requires a suitable GPU.")

In [ ]:
# ---------------- DATASET FINGERPRINT + PACKAGE ID ----------------
from qmini.config import QMiniConfig

base_config = QMiniConfig.from_json(MODEL_CONFIG_PATH)
if base_config.estimated_parameter_count() > 250_000_000:
    raise RuntimeError(
        "This notebook is deliberately limited to the <=250M Colab research profile. "
        "The 5B profile requires a real distributed trainer that this repository does not yet implement."
    )

if RELEASE_MODE and not SOURCE_METADATA_REL:
    raise ValueError("RELEASE_MODE=True requires SOURCE_METADATA_REL to point to a source/license manifest.")

SOURCE_METADATA_PATH = (MY_DRIVE / SOURCE_METADATA_REL) if SOURCE_METADATA_REL else None
if SOURCE_METADATA_PATH is not None and not SOURCE_METADATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset source metadata not found: {SOURCE_METADATA_PATH}")

data_files = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and not any(part.startswith(".") for part in path.relative_to(DATA_DIR).parts)
)
if not data_files:
    raise FileNotFoundError(
        f"No corpus files found in {DATA_DIR}. Put your reviewed text/JSONL/Parquet corpus there, then rerun."
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_records = []
for path in data_files:
    source_records.append({
        "relative_path": path.relative_to(DATA_DIR).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

source_metadata_sha256 = (
    sha256_file(SOURCE_METADATA_PATH) if SOURCE_METADATA_PATH is not None else None
)
package_identity = {
    "format": "qmini-training-ready-package-id-v1",
    "source_revision": SOURCE_REVISION,
    "model_config": base_config.to_dict(),
    "dataset_files": source_records,
    "source_metadata_sha256": source_metadata_sha256,
    "tokenizer": {
        "type": "byte-level-bpe",
        "vocab_size_target": VOCAB_SIZE,
        "validation_fraction": VALIDATION_FRACTION,
        "near_duplicate_hamming_distance": NEAR_DUPLICATE_HAMMING_DISTANCE,
    },
    "preparation": {
        "loss_mask_mode": LOSS_MASK_MODE,
        "minimum_sequence_length": TRAIN_SEQUENCE_LENGTH,
    },
    "release_mode": RELEASE_MODE,
}
PACKAGE_KEY = hashlib.sha256(
    json.dumps(package_identity, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]

PACKAGE_NAME = f"QMini-152M-Q2v2-{PACKAGE_KEY}"
LOCAL_PACKAGE_DIR = Path("/content") / PACKAGE_NAME
FINAL_PACKAGE_DIR = ART_ROOT / "training_ready" / PACKAGE_NAME
FINAL_PACKAGE_DIR.parent.mkdir(parents=True, exist_ok=True)

print("Corpus files:", len(source_records))
print("Corpus bytes:", f"{sum(item['size_bytes'] for item in source_records):,}")
print("Package identity:", PACKAGE_KEY)
print("Final Drive package:", FINAL_PACKAGE_DIR)

In [ ]:
# ---------------- TOKENIZER + TOKEN STREAMS + FULL MODEL BUILD ----------------
from qmini.tokenizer import load_tokenizer
from qmini.text import SPECIAL_TOKENS
from qmini.export import verify_packed_export
from qmini.train import (
    _verify_initial_brain_artifact,
    _verify_prepared_dataset_manifest,
    _load_tokens,
    _validate_token_ids,
)

def verify_package(package_dir):
    package_dir = Path(package_dir)
    tokenizer_path = package_dir / "tokenizer" / "tokenizer.json"
    tokenizer_sidecar = Path(str(tokenizer_path) + ".qmini.json")
    dataset_dir = package_dir / "dataset"
    initial_dir = package_dir / "initial_brain"
    config_path = initial_dir / "config.json"
    model_path = initial_dir / "model.qmini.npz"
    manifest_path = initial_dir / "model_manifest.json"

    active_config_path = package_dir / "active_config.json"
    required = [
        tokenizer_path, tokenizer_sidecar, active_config_path, dataset_dir / "train.bin",
        dataset_dir / "valid.bin", dataset_dir / "train.mask.bin",
        dataset_dir / "valid.mask.bin", dataset_dir / "dataset.json",
        config_path, model_path, manifest_path,
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Training-ready package is incomplete: " + ", ".join(missing))

    config = QMiniConfig.from_json(config_path)
    if QMiniConfig.from_json(active_config_path).to_dict() != config.to_dict():
        raise RuntimeError("Root active_config.json differs from initial_brain/config.json.")
    tokenizer = load_tokenizer(tokenizer_path)
    if tokenizer.get_vocab_size() != config.vocab_size:
        raise RuntimeError("Tokenizer vocabulary size does not match the model config.")
    for token in SPECIAL_TOKENS:
        if tokenizer.token_to_id(token) is None:
            raise RuntimeError(f"Tokenizer is missing required structural token: {token}")

    packed_report = verify_packed_export(model_path)
    if (
        packed_report.get("format") != "qmini-packed-v2"
        or packed_report.get("q2_version") != 2
        or packed_report.get("parameter_count") != config.estimated_parameter_count()
    ):
        raise RuntimeError("Q2-v2 archive failed format/version/parameter verification.")

    _verify_initial_brain_artifact(model_path, config)
    _verify_prepared_dataset_manifest(
        dataset_dir / "train.bin", dataset_dir / "valid.bin",
        tokenizer_path, dataset_dir / "train.mask.bin",
        dataset_dir / "valid.mask.bin", config.vocab_size,
    )
    train_tokens = _load_tokens(dataset_dir / "train.bin", "Training")
    valid_tokens = _load_tokens(dataset_dir / "valid.bin", "Validation")
    _validate_token_ids(train_tokens, config.vocab_size, "Training")
    _validate_token_ids(valid_tokens, config.vocab_size, "Validation")
    if len(train_tokens) < TRAIN_SEQUENCE_LENGTH + 2:
        raise ValueError("Training corpus is too short for TRAIN_SEQUENCE_LENGTH.")
    if len(valid_tokens) < TRAIN_SEQUENCE_LENGTH + 2:
        raise ValueError("Validation corpus is too short for TRAIN_SEQUENCE_LENGTH.")

    model_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if model_manifest.get("training_steps") != 0 or model_manifest.get("optimizer_steps") != 0:
        raise RuntimeError("Initial brain unexpectedly contains model-training updates.")
    if model_manifest.get("matrix_weight_bits") != 2:
        raise RuntimeError("Initial brain is not a Q2-v2 two-bit matrix-weight artifact.")

    return {
        "config": config,
        "packed_report": packed_report,
        "model_manifest": model_manifest,
        "tokenizer_sha256": sha256_file(tokenizer_path),
        "dataset_manifest": json.loads((dataset_dir / "dataset.json").read_text(encoding="utf-8")),
        "train_tokens": len(train_tokens),
        "valid_tokens": len(valid_tokens),
    }

if FINAL_PACKAGE_DIR.is_dir():
    print("A package with this exact source/config fingerprint already exists. Verifying; no overwrite will occur.")
    PACKAGE_DIR = FINAL_PACKAGE_DIR
    verified = verify_package(PACKAGE_DIR)
else:
    if LOCAL_PACKAGE_DIR.exists():
        raise FileExistsError(
            f"Local staging folder already exists: {LOCAL_PACKAGE_DIR}. "
            "Reconnect to a fresh Colab runtime rather than deleting an unverified build."
        )

    LOCAL_PACKAGE_DIR.mkdir(parents=True, exist_ok=False)
    TOKENIZER_DIR = LOCAL_PACKAGE_DIR / "tokenizer"
    DATASET_DIR = LOCAL_PACKAGE_DIR / "dataset"
    INITIAL_DIR = LOCAL_PACKAGE_DIR / "initial_brain"
    TOKENIZER_DIR.mkdir()

    TOKENIZER_PATH = TOKENIZER_DIR / "tokenizer.json"
    TOKENIZER_CONFIG_PATH = LOCAL_PACKAGE_DIR / "active_config.json"

    tokenizer_command = [
        sys.executable, "-m", "qmini.cli", "tokenizer",
        "--data-dir", str(DATA_DIR),
        "--output", str(TOKENIZER_PATH),
        "--vocab-size", str(VOCAB_SIZE),
        "--base-config", str(MODEL_CONFIG_PATH),
        "--valid-fraction", str(VALIDATION_FRACTION),
        "--near-duplicate-hamming-distance", str(NEAR_DUPLICATE_HAMMING_DISTANCE),
        "--config-output", str(TOKENIZER_CONFIG_PATH),
    ]
    if SOURCE_METADATA_PATH is not None:
        tokenizer_command.extend(["--source-metadata", str(SOURCE_METADATA_PATH)])
    if RELEASE_MODE:
        tokenizer_command.append("--release-mode")
    run_checked(tokenizer_command, cwd=REPO_DIR)

    active_config = QMiniConfig.from_json(TOKENIZER_CONFIG_PATH)
    active_values = active_config.to_dict()
    base_values = base_config.to_dict()
    base_values["vocab_size"] = active_config.vocab_size
    if active_values != base_values:
        raise RuntimeError("Tokenizer creation unexpectedly changed architecture fields other than vocab_size.")

    tokenizer = load_tokenizer(TOKENIZER_PATH)
    if tokenizer.get_vocab_size() != active_config.vocab_size:
        raise RuntimeError("Tokenizer and active model config vocabulary do not match.")
    for token in SPECIAL_TOKENS:
        if tokenizer.token_to_id(token) is None:
            raise RuntimeError(f"Tokenizer is missing required structural token: {token}")

    prepare_command = [
        sys.executable, "-m", "qmini.cli", "prepare",
        "--data-dir", str(DATA_DIR),
        "--tokenizer", str(TOKENIZER_PATH),
        "--output-dir", str(DATASET_DIR),
        "--valid-fraction", str(VALIDATION_FRACTION),
        "--near-duplicate-hamming-distance", str(NEAR_DUPLICATE_HAMMING_DISTANCE),
        "--loss-mask", LOSS_MASK_MODE,
    ]
    if SOURCE_METADATA_PATH is not None:
        prepare_command.extend(["--source-metadata", str(SOURCE_METADATA_PATH)])
    if RELEASE_MODE:
        prepare_command.append("--release-mode")
    run_checked(prepare_command, cwd=REPO_DIR)

    run_checked([
        sys.executable, "-m", "qmini.cli", "audit",
        "--config", str(TOKENIZER_CONFIG_PATH),
    ], cwd=REPO_DIR, capture_output=False)

    parameter_count = active_config.estimated_parameter_count()
    if parameter_count > 250_000_000:
        raise RuntimeError("The selected profile exceeds this notebook's single-device Colab limit.")

    LOCAL_INITIAL_DIR = LOCAL_PACKAGE_DIR / "initial_brain"
    run_checked([
        sys.executable, "-m", "qmini.cli", "build-initial",
        "--config", str(TOKENIZER_CONFIG_PATH),
        "--output-dir", str(LOCAL_INITIAL_DIR),
        "--source-revision", SOURCE_REVISION,
    ], cwd=REPO_DIR)

    # Bind the initial weights to the exact tokenizer and prepared corpus.
    manifest_path = LOCAL_INITIAL_DIR / "model_manifest.json"
    model_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    dataset_manifest_path = DATASET_DIR / "dataset.json"
    dataset_manifest = json.loads(dataset_manifest_path.read_text(encoding="utf-8"))
    model_manifest.update({
        "tokenizer_included": True,
        "tokenizer_relative_path": "../tokenizer/tokenizer.json",
        "tokenizer_sha256": sha256_file(TOKENIZER_PATH),
        "tokenizer_manifest_sha256": sha256_file(Path(str(TOKENIZER_PATH) + ".qmini.json")),
        "prepared_dataset_relative_path": "../dataset",
        "prepared_dataset_manifest_sha256": sha256_file(dataset_manifest_path),
        "train_tokens_sha256": dataset_manifest["train_tokens_sha256"],
        "valid_tokens_sha256": dataset_manifest["valid_tokens_sha256"],
        "train_mask_sha256": dataset_manifest.get("train_mask_sha256"),
        "valid_mask_sha256": dataset_manifest.get("valid_mask_sha256"),
        "corpus_sha256": dataset_manifest.get("corpus_sha256"),
        "source_provenance_sha256": dataset_manifest.get("source_provenance_sha256"),
        "release_mode": bool(dataset_manifest.get("release_mode", False)),
    })
    manifest_path.write_text(
        json.dumps(model_manifest, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )

    # Add an inspectable build report without changing model weights.
    build_report = {
        "format": "qmini-training-ready-build-report-v1",
        "package_key": PACKAGE_KEY,
        "source_revision": SOURCE_REVISION,
        "config": active_config.to_dict(),
        "parameters": parameter_count,
        "architecture": {
            "causal_depthwise_convolution": True,
            "gated_diagonal_state_space": bool(active_config.use_state_space),
            "sparse_top1_moe": active_config.n_experts > 1 and active_config.top_k == 1,
            "shared_refinement_block": True,
            "transformer_self_attention": False,
            "qkv_attention": False,
        },
        "initial_model": {
            "format": "qmini-packed-v2",
            "matrix_weight_bits": 2,
            "sha256": sha256_file(LOCAL_INITIAL_DIR / "model.qmini.npz"),
            "bytes": (LOCAL_INITIAL_DIR / "model.qmini.npz").stat().st_size,
        },
        "tokenizer": {
            "path": "tokenizer/tokenizer.json",
            "vocabulary_size": tokenizer.get_vocab_size(),
            "sha256": sha256_file(TOKENIZER_PATH),
        },
        "dataset": {
            "path": "dataset/",
            "manifest_sha256": sha256_file(dataset_manifest_path),
            "train_tokens": int(dataset_manifest["train_tokens"]),
            "valid_tokens": int(dataset_manifest["valid_tokens"]),
            "loss_mask_mode": dataset_manifest["loss_mask_mode"],
            "release_mode": bool(dataset_manifest.get("release_mode", False)),
        },
        "source_corpus_fingerprint": PACKAGE_KEY,
        "training": {
            "started": False,
            "training_steps": 0,
            "optimizer_steps": 0,
            "tokens_seen": 0,
        },
        "precision": {
            "matrix_weights": "Q2-v2 / 2-bit symbols",
            "row_scales": "FP32",
            "one_dimensional_parameters": "FP16 in packed export",
            "later_QAT_master_parameters": "FP32",
            "later_AdamW_state": "FP32",
            "activations": "signed INT8 fake quantization",
        },
    }
    (LOCAL_PACKAGE_DIR / "BUILD_REPORT.json").write_text(
        json.dumps(build_report, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    # Validate the local package before copying it to Drive.
    verified = verify_package(LOCAL_PACKAGE_DIR)

    # Publish through a staging directory. Never overwrite an earlier package.
    STAGING_DIR = FINAL_PACKAGE_DIR.parent / f".{PACKAGE_NAME}.staging-{int(time.time())}"
    if STAGING_DIR.exists():
        raise FileExistsError(f"Staging directory already exists: {STAGING_DIR}")
    shutil.copytree(LOCAL_PACKAGE_DIR, STAGING_DIR)
    verify_package(STAGING_DIR)
    STAGING_DIR.rename(FINAL_PACKAGE_DIR)
    PACKAGE_DIR = FINAL_PACKAGE_DIR

# Re-verify the published Drive copy, not just the local source.
verified = verify_package(PACKAGE_DIR)
print("\nQ-MINI TRAINING-READY PACKAGE VERIFIED")
print("Drive package:", PACKAGE_DIR)
print("Parameters:", f"{verified['config'].estimated_parameter_count():,}")
print("Tokenizer vocabulary:", verified["config"].vocab_size)
print("Q2-v2 artifact bytes:", verified["packed_report"].get("archive_bytes"))
print("Training tokens:", f"{verified['train_tokens']:,}")
print("Validation tokens:", f"{verified['valid_tokens']:,}")
print("Neural optimizer steps:", verified["model_manifest"]["optimizer_steps"])
print("Model weight updates:", verified["model_manifest"]["training_steps"])
print("No neural-model training was run in this build stage.")

In [ ]:
# ---------------- OPTIONAL LATER TRAINING ----------------
# Keep RUN_TRAINING_NOW=False during the build-only session.
# To start pretraining in a later session, reopen this notebook, mount Drive,
# set RUN_TRAINING_NOW=True in the settings cell, then run cells top-to-bottom.

from qmini.config import QMiniConfig

INITIAL_DIR = PACKAGE_DIR / "initial_brain"
DATASET_DIR = PACKAGE_DIR / "dataset"
TOKENIZER_PATH = PACKAGE_DIR / "tokenizer" / "tokenizer.json"
CONFIG_PATH = INITIAL_DIR / "config.json"
INITIAL_MODEL_PATH = INITIAL_DIR / "model.qmini.npz"

active_config = QMiniConfig.from_json(CONFIG_PATH)
if TRAIN_SEQUENCE_LENGTH < 2 or TRAIN_SEQUENCE_LENGTH > active_config.max_context:
    raise ValueError("TRAIN_SEQUENCE_LENGTH must be in [2, config.max_context].")
if TRAINING_STEPS < 1:
    raise ValueError("TRAINING_STEPS must be a positive total optimizer-step target.")

TRAIN_RUN_DIR = ART_ROOT / "training_runs" / f"QMini-152M-{PACKAGE_KEY}"
checkpoint_path = TRAIN_RUN_DIR / "latest.pt"
final_manifest_path = TRAIN_RUN_DIR / "final_model" / "model_manifest.json"

train_command = [
    sys.executable, "-m", "qmini.cli", "train",
    "--config", str(CONFIG_PATH),
    "--train-bin", str(DATASET_DIR / "train.bin"),
    "--valid-bin", str(DATASET_DIR / "valid.bin"),
    "--train-mask-bin", str(DATASET_DIR / "train.mask.bin"),
    "--valid-mask-bin", str(DATASET_DIR / "valid.mask.bin"),
    "--tokenizer", str(TOKENIZER_PATH),
    "--output-dir", str(TRAIN_RUN_DIR),
    "--steps", str(TRAINING_STEPS),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--seq-len", str(TRAIN_SEQUENCE_LENGTH),
    "--grad-accum", str(TRAIN_GRAD_ACCUM),
    "--lr", str(TRAIN_LEARNING_RATE),
    "--weight-decay", str(TRAIN_WEIGHT_DECAY),
    "--eval-every", str(TRAIN_EVAL_EVERY),
    "--save-every", str(TRAIN_SAVE_EVERY),
    "--eval-batches", str(TRAIN_EVAL_BATCHES),
]

if not RUN_TRAINING_NOW:
    import shlex
    print("BUILD-ONLY MODE: no optimizer was created and no model weights were updated.")
    print("Training-ready package:", PACKAGE_DIR)
    print("To train later, set RUN_TRAINING_NOW=True and rerun this cell after reviewing the GPU budget.")
    print("The first training run will use --init-from-initial with the verified Q2-v2 model.")
    print(shlex.join(train_command + ["--init-from-initial", str(INITIAL_MODEL_PATH)]))
else:
    if not torch.cuda.is_available():
        raise RuntimeError("Neural-model training requires a CUDA GPU. Build-only mode remains available on CPU.")
    free_gpu_bytes, total_gpu_bytes = torch.cuda.mem_get_info(0)
    print("Training GPU:", torch.cuda.get_device_name(0))
    print(f"GPU free: {free_gpu_bytes / 1024**3:.2f} GiB / total: {total_gpu_bytes / 1024**3:.2f} GiB")
    if free_gpu_bytes < TRAIN_MIN_FREE_GPU_GIB * (1024 ** 3):
        raise RuntimeError(
            f"Training blocked: free GPU memory is below {TRAIN_MIN_FREE_GPU_GIB:.1f} GiB. "
            "Choose a higher-memory GPU; lowering this guard does not guarantee a successful run."
        )

    TRAIN_RUN_DIR.mkdir(parents=True, exist_ok=True)
    if checkpoint_path.is_file():
        state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        if state.get("config") != active_config.to_dict():
            raise RuntimeError("Existing checkpoint config differs; it was preserved.")
        run_manifest_path = TRAIN_RUN_DIR / "run.json"
        if not run_manifest_path.is_file():
            raise RuntimeError("Existing checkpoint lacks run.json; refusing an unsafe resume.")
        run_manifest = json.loads(run_manifest_path.read_text(encoding="utf-8"))
        recorded_initial = run_manifest.get("initial_model") or {}
        expected_initial_sha = sha256_file(INITIAL_MODEL_PATH)
        if recorded_initial.get("sha256") != expected_initial_sha:
            raise RuntimeError("Existing run was not initialized from this Q2 brain; it was preserved.")

        checkpoint_step = int(state.get("step", 0))
        if checkpoint_step < TRAINING_STEPS:
            train_command.append("--resume")
            saved_schedule_steps = int(state.get("schedule_total_steps", TRAINING_STEPS))
            if TRAINING_STEPS > saved_schedule_steps:
                train_command.append("--extend-schedule")
            run_checked(train_command)
        elif not final_manifest_path.is_file():
            train_command[train_command.index("--steps") + 1] = str(checkpoint_step + 1)
            train_command.extend(["--resume", "--extend-schedule"])
            run_checked(train_command)
        else:
            print("Requested training target is already complete:", final_manifest_path)
    else:
        train_command.extend(["--init-from-initial", str(INITIAL_MODEL_PATH)])
        run_checked(train_command)

    if not final_manifest_path.is_file():
        raise RuntimeError("Training command ended without a verified final_model manifest.")
    print("Q-Mini pretraining output:", TRAIN_RUN_DIR / "final_model")
    print("Resumable training checkpoint:", checkpoint_path)

## Precision and readiness boundary

The packed initial artifact stores **matrix-weight symbols in Q2-v2 (2 bits per symbol)**. Its row scales are FP32 and one-dimensional parameters are FP16. During later QAT, the current trainer uses FP32 master parameters and FP32 AdamW optimizer state; this is not an all-state 2-bit training system.

The 152M-class model is the supported single-device research path, subject to actual free GPU memory and Colab session limits. The 5B model has no implemented FSDP/ZeRO trainer yet. No notebook can guarantee that an arbitrary Colab session will have enough VRAM or uninterrupted runtime.

The package is a complete **untrained starting point**: model architecture, Q2 initial weights, tokenizer, prepared training/validation streams, response masks, config, manifest and hashes. It is not already a capable conversational AI. Its language ability is developed only by subsequent training and must be measured on independent held-out data.